# Checagens da base e validação externa (8 espécies)

Este notebook reúne o código de dois grupos de resultados do artigo que dependem de
informações que não entram na rede (local, data e hora de cada gravação):

- **Parte 1 — Checagens da base:** projeto de origem de cada espécie, gravações do teste com
  uma "vizinha" no treino (mesmo local, menos de 60 s de diferença) e uma divisão em que
  noites inteiras ficam fora do treino;
- **Parte 2 — Validação externa:** gravações que entraram na planilha original depois de o
  modelo estar pronto (cópias de 01/10/2026 e de 11/10/2026), avaliadas sem nenhum ajuste.

A rede e o treino são os do notebook `10`. O modelo final não é alterado; nada aqui é usado
para escolher hiperparâmetros.

As saídas mostram só contagens e métricas. Local, data e nome de arquivo são usados nas
contas, mas não são impressos.

In [1]:
import copy
import re
import zipfile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

In [2]:
# Fixando seeds para poder reproduzir os resultados
np.random.seed(42)
torch.manual_seed(42)

# Abas de identificação da planilha usada no trabalho (exportadas em CSV)
ARQUIVO = "../banco/Cidade Viva e PELG Bioacústica (OFICIAL).xlsx - {}.csv"

# Cópias da planilha feitas depois de o modelo estar pronto
PLANILHA_0110 = "../Cópia de Cidade Viva e PELG Bioacústica (OFICIAL) - 01_10.xlsx"
PLANILHA_1110 = "../Cidade Viva e PELG Bioacústica (OFICIAL)_11-10-26.xlsx"

COL_FMIN, COL_FMAX, COL_FME, COL_DUR = "Fmin (kHz)", "Fmax (kHz)", "FME (kHz)", "Duração (ms)"
MEDIDAS_PLANILHA = [COL_FMIN, COL_FMAX, COL_FME, COL_DUR]

## Base, divisão e normalização

Mesmos passos do notebook `10` (recorte das 8 espécies, 5 erros graves, duplicatas, divisão
estratificada 70/15/15 com semente 42, normalização com média e desvio do treino). A diferença
é que aqui a leitura parte das abas originais, que têm local, data e hora.

In [3]:
aba_cv = pd.read_csv(ARQUIVO.format("Cidade Viva - Identificação"))
aba_pelg = pd.read_csv(ARQUIVO.format("PELG - Identificação"))
identificacao = pd.concat([aba_cv, aba_pelg], ignore_index=True)


# Padroniza o nome do arquivo para reconhecer gravações repetidas
def normaliza_arquivo(nome):
    if pd.isna(nome):
        return nome
    nome = str(nome).strip()
    # tira a anotação entre parênteses
    nome = nome.split("(")[0].strip()
    # tira o sufixo _000
    return re.sub(r"_000$", "", nome)


identificacao["Arquivo_norm"] = identificacao["Arquivo"].apply(normaliza_arquivo)

# Espécies com pelo menos 379 gravações (o sonótipo Molossidae1 fica de fora)
contagem = identificacao["Espécie/Sonótipo"].value_counts().drop("Molossidae1")
especies_recorte = contagem[contagem >= 379].index.tolist()
dados = identificacao[identificacao["Espécie/Sonótipo"].isin(especies_recorte)].copy()

# Erros graves e duplicatas exatas
erro_lavali = (dados["Espécie/Sonótipo"] == "Myotis_lavali") & (dados[COL_FME] < 10)
erro_eumops = (dados["Espécie/Sonótipo"] == "Eumops_sp") & (dados[COL_FME] > 45)
dados = dados[~(erro_lavali | erro_eumops)]
dados = dados.drop_duplicates(subset=["Arquivo_norm", "Espécie/Sonótipo"], keep="first")

X_bruto = np.column_stack([
    dados[COL_FMIN].to_numpy(),
    dados[COL_FMAX].to_numpy(),
    (dados[COL_FMAX] - dados[COL_FMIN]).to_numpy(),  # largura de banda recalculada
    dados[COL_FME].to_numpy(),
    dados[COL_DUR].to_numpy(),
])
especie = dados["Espécie/Sonótipo"].to_numpy()
ESPECIES = pd.Series(especie).value_counts().index.tolist()
d_np = np.array([ESPECIES.index(e) for e in especie])
N_classes = len(ESPECIES)

# Projeto de origem: as linhas da primeira aba vêm antes das da segunda
projeto = np.where(dados.index.to_numpy() < len(aba_cv), "Cidade Viva", "PELG")

print("Gravações após a limpeza:", len(dados), "| espécies:", N_classes)

Gravações após a limpeza: 13408 | espécies: 8


In [4]:
# Divide cada espécie separadamente, para manter a proporção nos três conjuntos
def divisao_estratificada(rotulos, indices, proporcoes=(0.70, 0.15, 0.15), semente=42):
    rng = np.random.default_rng(semente)
    treino, val, teste = [], [], []
    for classe in np.unique(rotulos):
        idx = rng.permutation(indices[rotulos == classe])
        n_tr = int(round(len(idx) * proporcoes[0]))
        n_va = int(round(len(idx) * proporcoes[1]))
        treino.append(idx[:n_tr])
        val.append(idx[n_tr:n_tr + n_va])
        teste.append(idx[n_tr + n_va:])
    return np.concatenate(treino), np.concatenate(val), np.concatenate(teste)


idx_tr, idx_va, idx_te = divisao_estratificada(especie, dados.index.to_numpy())
pos_tr, pos_va, pos_te = (dados.index.get_indexer(i) for i in (idx_tr, idx_va, idx_te))

# Normalização com média e desvio do treino
X_mean = X_bruto[pos_tr].mean(axis=0)
X_std = X_bruto[pos_tr].std(axis=0)
X_np = ((X_bruto - X_mean) / X_std).astype(np.float32)

# Conferência: é a mesma base preparada no notebook 02 e usada nos notebooks 10 e 12?
preparado = np.load("../dados_processados/etapa2.npz", allow_pickle=True)
print("treino / validação / teste:", len(pos_tr), len(pos_va), len(pos_te))
print("entradas iguais às do notebook 02:", np.array_equal(X_np, preparado["X_simples"].astype(np.float32)))
print("mesma divisão:", np.array_equal(pos_tr, preparado["pos_treino_simples"]),
      np.array_equal(pos_va, preparado["pos_val_simples"]), np.array_equal(pos_te, preparado["pos_teste_simples"]))

treino / validação / teste: 9386 2011 2011
entradas iguais às do notebook 02: True
mesma divisão: True True True


## Rede, treino e medidas

Os mesmos do notebook `10`. A função de treino recebe os conjuntos de treino e de validação,
porque a divisão por noite (Parte 1) usa conjuntos diferentes dos habituais; o peso de cada
espécie na função custo é calculado com o treino recebido.

In [5]:
# Ajuste de hiperparâmetros

# passo de adaptação
eta = 0.01

# Tamanho do mini-batch
Nb = 64

# Número de épocas
Ne = 1000

# Sementes dos três treinos
sementes = [0, 1, 2]


class Model(nn.Module):
    # N_entradas: quantas entradas a rede recebe
    def __init__(self, N_entradas):
        super().__init__()
        self.model = nn.Sequential(
            # Entradas conectadas a 8 neurônios
            nn.Linear(N_entradas, 8),
            # Função de ativação tangente hiperbólica
            nn.Tanh(),
            # Saídas dos 8 neurônios conectadas a 8 neurônios de saída (um por espécie)
            nn.Linear(8, N_classes),
        )

    def forward(self, x):
        output = self.model(x)
        return output

In [6]:
# Treina uma rede e devolve o modelo com os pesos da época de menor perda de validação.
# É o mesmo loop do notebook 10.
def treina_rede(Xtr_np, dtr_np, Xva_np, dva_np, semente):
    Xtr = torch.tensor(Xtr_np, dtype=torch.float32)
    dtr = torch.tensor(dtr_np).type(torch.LongTensor)
    Xva = torch.tensor(Xva_np, dtype=torch.float32)
    dva = torch.tensor(dva_np).type(torch.LongTensor)
    train_set = [(Xtr[i, :], dtr[i]) for i in range(Xtr.shape[0])]

    # Peso de cada espécie na função custo: inverso da frequência no treino
    contagem_treino = np.bincount(dtr_np, minlength=N_classes)
    pesos = len(dtr_np) / (N_classes * np.maximum(contagem_treino, 1))
    loss_function = nn.CrossEntropyLoss(weight=torch.tensor(pesos, dtype=torch.float32))

    # A semente define a inicialização dos pesos...
    torch.manual_seed(semente)
    model = Model(Xtr.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=eta)

    # ...e a ordem dos mini-batches
    torch.manual_seed(semente)
    train_loader = torch.utils.data.DataLoader(train_set, batch_size=Nb, shuffle=True)

    melhor_val_loss = float("inf")
    melhores_pesos = None
    melhor_epoca = -1

    # Loop das épocas
    for epoch in range(Ne):
        model.train()
        # Loop dos mini-batches
        for X, d in train_loader:
            model.zero_grad()
            y = model(X)
            loss = loss_function(y, d)
            loss.backward()
            optimizer.step()

        # Validação: modelo em modo de inferência e sem cálculo de gradientes
        model.eval()
        with torch.no_grad():
            y_val = model(Xva)
            val_loss = loss_function(y_val, dva)

        # Guarda os pesos se esta for a menor perda de validação até agora
        if val_loss.item() < melhor_val_loss:
            melhor_val_loss = val_loss.item()
            melhores_pesos = copy.deepcopy(model.state_dict())
            melhor_epoca = epoch + 1

    model.load_state_dict(melhores_pesos)
    model.eval()
    return model, melhor_epoca


# Espécie prevista: a de maior probabilidade na saída
def prever(model, X_entrada):
    model.eval()
    with torch.no_grad():
        y = model(torch.tensor(X_entrada, dtype=torch.float32))
        prob = torch.softmax(y, dim=1)
        _, predito = torch.max(prob, 1)
    return predito.numpy()


# Matriz de confusão: linha = espécie prevista, coluna = espécie verdadeira
def matriz_confusao(d, predito):
    M = np.zeros((N_classes, N_classes), dtype=int)
    for i in range(len(d)):
        M[predito[i], d[i]] += 1
    return M


# Acurácia balanceada (média das sensibilidades das espécies presentes), acurácia e sensibilidades.
# Nas gravações externas algumas espécies têm poucas gravações ou nenhuma; as ausentes ficam de fora.
def medidas(M):
    sens = np.full(N_classes, np.nan)
    for k in range(N_classes):
        if M[:, k].sum() > 0:
            sens[k] = M[k, k] / M[:, k].sum()
    acuracia_balanceada = np.nanmean(sens)
    acuracia = np.trace(M) / M.sum()
    return acuracia_balanceada, acuracia, sens

## Modelos

A rede final (5 medidas) e a variante de 3 medidas (Fmin, Fmax e FME) do notebook `12`, três
sementes cada. As épocas guardadas devem ser as mesmas daqueles notebooks.

In [7]:
variantes = {"5 medidas": [0, 1, 2, 3, 4], "3 medidas": [0, 1, 3]}  # colunas de X_np

modelos = {}
for nome, colunas in variantes.items():
    for semente in sementes:
        model, melhor_epoca = treina_rede(X_np[pos_tr][:, colunas], d_np[pos_tr],
                                          X_np[pos_va][:, colunas], d_np[pos_va], semente)
        modelos[(nome, semente)] = model
        print(f"{nome} - semente {semente}: pesos da época {melhor_epoca}")

5 medidas - semente 0: pesos da época 818


5 medidas - semente 1: pesos da época 339


5 medidas - semente 2: pesos da época 438


3 medidas - semente 0: pesos da época 741


3 medidas - semente 1: pesos da época 445


3 medidas - semente 2: pesos da época 926


In [8]:
# Conferência no conjunto de teste (mesmos números dos notebooks 10 e 12)
for nome, colunas in variantes.items():
    ab = []
    for semente in sementes:
        M = matriz_confusao(d_np[pos_te], prever(modelos[(nome, semente)], X_np[pos_te][:, colunas]))
        ab.append(medidas(M)[0])
    print(f"{nome} - teste: acurácia balanceada {np.mean(ab):.3f} ± {np.std(ab):.4f}")

# Previsões da rede final, semente 0, usadas na Parte 1
predito_teste = prever(modelos[("5 medidas", 0)], X_np[pos_te])
print("rede final, semente 0, acurácia balanceada no teste:",
      round(medidas(matriz_confusao(d_np[pos_te], predito_teste))[0], 4))

5 medidas - teste: acurácia balanceada 0.959 ± 0.0002
3 medidas - teste: acurácia balanceada 0.959 ± 0.0011
rede final, semente 0, acurácia balanceada no teste: 0.9591


---
# Parte 1 — Checagens da base

## Espécie e projeto de origem

As gravações vêm de dois projetos. Se uma espécie vem quase toda de um deles, a rede poderia
estar reconhecendo o projeto, e não a espécie; os dois efeitos não podem ser separados.

In [9]:
print(f"{'espécie':24s} {'Cidade Viva':>12s} {'PELG':>8s}")
for k in range(N_classes):
    da_especie = d_np == k
    print(f"{ESPECIES[k]:24s} {np.sum(da_especie & (projeto == 'Cidade Viva')):12d} {np.sum(da_especie & (projeto == 'PELG')):8d}")

# FME média de Molossus molossus em cada projeto
k = ESPECIES.index("Molossus_molossus")
print("\nFME média de Molossus molossus (kHz):")
for p in ["Cidade Viva", "PELG"]:
    m = (d_np == k) & (projeto == p)
    print(f"   {p}: {X_bruto[m, 3].mean():.1f} (n = {m.sum()})")

espécie                   Cidade Viva     PELG
Myotis_lavali                    4331     2953
Molossus_molossus                 708      940
Lasiurus_blossevillii            1061       21
Neoeptesicus_furinalis            975       39
Eumops_sp                         117      606
Molossus_currentium               715        1
Molossops_temminckii              427      130
Myotis_nigricans                  375        9

FME média de Molossus molossus (kHz):
   Cidade Viva: 34.5 (n = 708)
   PELG: 40.1 (n = 940)


## Gravações do teste com uma "vizinha" no treino

Gravações feitas com poucos segundos de diferença no mesmo local costumam ser do mesmo
animal. Na divisão aleatória, uma delas pode ir para o treino e outra para o teste. Aqui se
conta quantas gravações do teste têm uma vizinha no treino (mesmo local, menos de 60 s de
diferença) e se a rede acerta mais nelas.

In [10]:
# Data e hora de cada gravação (usadas só nas contas; não são impressas)
data_hora = pd.to_datetime(dados["Data"].astype(str).str.replace("_", "/") + " " + dados["Hora"].astype(str),
                           dayfirst=True, errors="coerce").to_numpy()
area = dados["Área"].fillna("?").to_numpy()
print(f"data e hora interpretadas: {np.mean(~pd.isna(data_hora)) * 100:.1f}% das gravações")

treino = pd.DataFrame({"t": data_hora[pos_tr], "a": area[pos_tr], "d": d_np[pos_tr]}).dropna()

tem_vizinha = []         # vizinha de qualquer espécie
tem_vizinha_mesma = []   # vizinha da mesma espécie
for p in pos_te:
    if pd.isna(data_hora[p]):
        tem_vizinha.append(False)
        tem_vizinha_mesma.append(False)
        continue
    perto = treino[(treino.a == area[p]) & (abs((treino.t - data_hora[p]).dt.total_seconds()) < 60)]
    tem_vizinha.append(len(perto) > 0)
    tem_vizinha_mesma.append(bool((perto.d == d_np[p]).any()))
tem_vizinha = np.array(tem_vizinha)
tem_vizinha_mesma = np.array(tem_vizinha_mesma)

d_teste = d_np[pos_te]
print(f"gravações do teste com vizinha no treino: {tem_vizinha.mean() * 100:.1f}% "
      f"(da mesma espécie: {tem_vizinha_mesma.mean() * 100:.1f}%)")
print(f"acurácia no teste, com vizinha da mesma espécie: {np.mean(predito_teste[tem_vizinha_mesma] == d_teste[tem_vizinha_mesma]):.3f} "
      f"(n = {tem_vizinha_mesma.sum()})")
print(f"acurácia no teste, sem vizinha: {np.mean(predito_teste[~tem_vizinha] == d_teste[~tem_vizinha]):.3f} "
      f"(n = {(~tem_vizinha).sum()})")

data e hora interpretadas: 100.0% das gravações


gravações do teste com vizinha no treino: 59.9% (da mesma espécie: 53.8%)
acurácia no teste, com vizinha da mesma espécie: 0.965 (n = 1082)
acurácia no teste, sem vizinha: 0.955 (n = 806)


## Divisão por noite

Uma checagem mais forte: noites inteiras (local e data) ficam fora do treino, de modo que
nenhuma gravação avaliada tem vizinha no treino. Usa só as gravações de treino e validação;
o conjunto de teste não entra. A normalização e os pesos por classe são refeitos com o
treino desta divisão; a rede é a mesma (semente 0).

In [11]:
pos_tv = np.concatenate([pos_tr, pos_va])
noite = pd.Series(area[pos_tv]).astype(str) + "|" + pd.Series(data_hora[pos_tv]).dt.date.astype(str)

# 70% das noites para treino, 15% para validação e 15% para avaliação
rng = np.random.default_rng(42)
noites = rng.permutation(noite.unique())
corte1, corte2 = int(0.70 * len(noites)), int(0.85 * len(noites))
noites_tr, noites_va, noites_av = set(noites[:corte1]), set(noites[corte1:corte2]), set(noites[corte2:])
tr2 = pos_tv[noite.isin(noites_tr).to_numpy()]
va2 = pos_tv[noite.isin(noites_va).to_numpy()]
av2 = pos_tv[noite.isin(noites_av).to_numpy()]
print(f"noites: {len(noites)} | gravações de treino / validação / avaliação: {len(tr2)} / {len(va2)} / {len(av2)}")

# normalização refeita com o treino desta divisão
X2 = ((X_bruto - X_bruto[tr2].mean(axis=0)) / X_bruto[tr2].std(axis=0)).astype(np.float32)
model_noite, melhor_epoca = treina_rede(X2[tr2], d_np[tr2], X2[va2], d_np[va2], 0)
M = matriz_confusao(d_np[av2], prever(model_noite, X2[av2]))
print(f"acurácia balanceada com noites inteiras fora do treino: {medidas(M)[0]:.3f} (divisão aleatória, validação: 0,966)")

noites: 290 | gravações de treino / validação / avaliação: 7482 / 1801 / 2114


acurácia balanceada com noites inteiras fora do treino: 0.959 (divisão aleatória, validação: 0,966)


---
# Parte 2 — Validação externa

## Leitura das planilhas novas

As cópias novas estão em `.xlsx`. A função abaixo lê as abas de identificação direto do
arquivo (um `.xlsx` é um arquivo compactado com as abas em XML), sem depender de outra
biblioteca. Um cuidado: no `.xlsx` as medidas aparecem multiplicadas por 1000 (o ponto
decimal foi gravado como separador de milhar), então são divididas por 1000.

In [12]:
NS = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
REL = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"


# Número da coluna a partir da referência da célula ("C5" -> 2)
def numero_coluna(ref):
    n = 0
    for letra in re.match(r"[A-Z]+", ref).group(0):
        n = n * 26 + ord(letra) - 64
    return n - 1


# Lê as abas de identificação de um .xlsx e devolve uma tabela só
def le_planilha(arquivo):
    z = zipfile.ZipFile(arquivo)
    # textos compartilhados entre as células
    textos = ["".join(t.text or "" for t in si.iter("{%s}t" % NS["m"]))
              for si in ET.fromstring(z.read("xl/sharedStrings.xml")).findall("m:si", NS)]
    alvo = {r.get("Id"): r.get("Target") for r in ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))}
    abas = {s.get("name"): alvo[s.get(REL)] for s in ET.fromstring(z.read("xl/workbook.xml")).find("m:sheets", NS)}

    tabelas = []
    for nome in [a for a in abas if "Identifica" in a]:
        raiz = ET.fromstring(z.read("xl/" + abas[nome].lstrip("/").replace("xl/", "")))
        linhas = {}
        for linha in raiz.iter("{%s}row" % NS["m"]):
            valores = {}
            for cel in linha.findall("m:c", NS):
                v = cel.find("m:v", NS)
                if v is None:
                    embutido = cel.find("m:is", NS)
                    texto = "".join(t.text or "" for t in embutido.iter("{%s}t" % NS["m"])) if embutido is not None else None
                elif cel.get("t") == "s":
                    texto = textos[int(v.text)]
                else:
                    texto = v.text
                valores[numero_coluna(cel.get("r"))] = texto
            linhas[int(linha.get("r"))] = valores
        cabecalho = linhas[1]
        colunas = [cabecalho.get(i) or f"col{i}" for i in range(max(cabecalho) + 1)]
        tabelas.append(pd.DataFrame([[linhas[r].get(i) for i in range(len(colunas))] for r in sorted(linhas) if r > 1],
                                    columns=colunas))
    tabela = pd.concat(tabelas, ignore_index=True)

    for col in MEDIDAS_PLANILHA:
        tabela[col] = pd.to_numeric(tabela[col].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    # no .xlsx as medidas vêm multiplicadas por 1000
    escala = 1000 if tabela[COL_FME].median() > 1000 else 1
    tabela[MEDIDAS_PLANILHA] = tabela[MEDIDAS_PLANILHA] / escala
    tabela["Arquivo_norm"] = tabela["Arquivo"].apply(normaliza_arquivo)
    return tabela, escala


planilha_0110, escala_0110 = le_planilha(PLANILHA_0110)
planilha_1110, escala_1110 = le_planilha(PLANILHA_1110)
print(f"linhas nas abas de identificação: base do trabalho {len(identificacao)} | 01/10 {len(planilha_0110)} | 11/10 {len(planilha_1110)}")
print(f"medidas divididas por: {escala_0110} (01/10) e {escala_1110} (11/10)")

linhas nas abas de identificação: base do trabalho 14540 | 01/10 16353 | 11/10 16353
medidas divididas por: 1000 (01/10) e 1000 (11/10)


## Quais gravações são novas

Uma gravação da planilha de 11/10 entra na validação externa se:

- o nome do arquivo (padronizado) **não** aparece na base usada no trabalho;
- é de uma das 8 espécies e tem as quatro medidas;
- o vetor de medidas não é idêntico ao de alguma gravação da base (evita a mesma gravação
  com outro nome).

Depois elas são separadas em dois grupos: as que já estavam na cópia de 01/10 e as inéditas
de 11/10. Uma "noite" é um par de local e data; ela é nova se não aparece na base do trabalho.

In [13]:
# O que identifica a base usada no trabalho (as duas abas inteiras, antes do recorte)
arquivos_base = set(identificacao["Arquivo_norm"].dropna())
vetores_base = set(map(tuple, identificacao[MEDIDAS_PLANILHA].round(3).dropna().to_numpy()))
noites_base = set(zip(identificacao["Área"].astype(str), identificacao["Data"].astype(str)))

# O mesmo para a cópia de 01/10
arquivos_0110 = set(planilha_0110["Arquivo_norm"].dropna())
vetores_0110 = set(map(tuple, planilha_0110[MEDIDAS_PLANILHA].round(3).dropna().to_numpy()))


def gravacoes_novas(tabela, arquivos_conhecidos, vetores_conhecidos):
    nome_novo = ~tabela["Arquivo_norm"].isin(arquivos_conhecidos) & tabela["Arquivo_norm"].notna()
    das_8 = tabela["Espécie/Sonótipo"].isin(ESPECIES)
    com_medidas = tabela[MEDIDAS_PLANILHA].notna().all(axis=1)
    vetor_novo = ~tabela[MEDIDAS_PLANILHA].round(3).apply(tuple, axis=1).isin(vetores_conhecidos)
    return nome_novo & das_8 & com_medidas & vetor_novo


fora_da_base = gravacoes_novas(planilha_1110, arquivos_base, vetores_base)
ineditas = gravacoes_novas(planilha_1110, arquivos_base | arquivos_0110, vetores_base | vetores_0110)

externas = planilha_1110[fora_da_base].copy()
e_inedita = ineditas[fora_da_base].to_numpy()
noite_ext = list(zip(externas["Área"].astype(str), externas["Data"].astype(str)))
noite_nova = np.array([n not in noites_base for n in noite_ext])

d_ext = np.array([ESPECIES.index(e) for e in externas["Espécie/Sonótipo"]])

print("gravações novas (fora da base do trabalho):", len(externas))
print("   já presentes na cópia de 01/10:", int((~e_inedita).sum()), "| inéditas de 11/10:", int(e_inedita.sum()))
print("   de noites que não estão na base:", int(noite_nova.sum()),
      "| noites distintas:", len(set(n for n, nova in zip(noite_ext, noite_nova) if nova)))
print("   inéditas de 11/10: noites distintas:", len(set(n for n, ined in zip(noite_ext, e_inedita) if ined)))

print(f"\n{'espécie':24s} {'todas':>6s} {'noites novas':>13s} {'inéditas':>9s}")
for k in range(N_classes):
    print(f"{ESPECIES[k]:24s} {np.sum(d_ext == k):6d} {np.sum((d_ext == k) & noite_nova):13d} {np.sum((d_ext == k) & e_inedita):9d}")

gravações novas (fora da base do trabalho): 1058
   já presentes na cópia de 01/10: 908 | inéditas de 11/10: 150
   de noites que não estão na base: 946 | noites distintas: 4
   inéditas de 11/10: noites distintas: 2

espécie                   todas  noites novas  inéditas
Myotis_lavali               795           770       121
Molossus_molossus            19            11         0
Lasiurus_blossevillii         4             4         2
Neoeptesicus_furinalis       62            40        14
Eumops_sp                     4             3         0
Molossus_currentium          34            22         2
Molossops_temminckii        128            84         0
Myotis_nigricans             12            12        11


## Desempenho nas gravações novas

As entradas são normalizadas com a média e o desvio do **treino original**. Nenhum peso é
ajustado. A regra da chave (espécie cuja FME típica está mais perto da FME medida) serve de
comparação, como no teste.

In [14]:
# Mesmas 5 medidas, com a largura de banda recalculada, e a mesma normalização do treino
X_ext_bruto = np.column_stack([
    externas[COL_FMIN].to_numpy(),
    externas[COL_FMAX].to_numpy(),
    (externas[COL_FMAX] - externas[COL_FMIN]).to_numpy(),
    externas[COL_FME].to_numpy(),
    externas[COL_DUR].to_numpy(),
]).astype(np.float64)
X_ext = ((X_ext_bruto - X_mean) / X_std).astype(np.float32)

# Regra de FME da chave de identificação
FME_CHAVE = {"Eumops_sp": 18.0, "Molossus_currentium": 30.0, "Molossus_molossus": 34.0,
             "Neoeptesicus_furinalis": 39.0, "Lasiurus_blossevillii": 46.0,
             "Molossops_temminckii": 50.0, "Myotis_lavali": 50.5, "Myotis_nigricans": 53.5}
centros = np.array([FME_CHAVE[e] for e in ESPECIES])
predito_regra = np.zeros(len(d_ext), dtype=int)
for i in range(len(d_ext)):
    predito_regra[i] = np.argmin(np.abs(X_ext_bruto[i, 3] - centros))

conjuntos = [("todas as gravações novas", np.ones(len(d_ext), dtype=bool)),
             ("só noites que não estão na base", noite_nova),
             ("já presentes na cópia de 01/10", ~e_inedita),
             ("inéditas de 11/10", e_inedita)]

for titulo, m in conjuntos:
    print(f"{titulo}: n = {m.sum()}")
    for nome, colunas in variantes.items():
        ab, acc = [], []
        for semente in sementes:
            M = matriz_confusao(d_ext[m], prever(modelos[(nome, semente)], X_ext[m][:, colunas]))
            ab.append(medidas(M)[0])
            acc.append(medidas(M)[1])
        print(f"   MLP, {nome}: acurácia balanceada {np.mean(ab):.3f} ± {np.std(ab):.3f} | acurácia {np.mean(acc):.3f}"
              f" | por semente: {np.round(ab, 4)}")
    M = matriz_confusao(d_ext[m], predito_regra[m])
    print(f"   regra da chave: acurácia balanceada {medidas(M)[0]:.3f} | acurácia {medidas(M)[1]:.3f}")

todas as gravações novas: n = 1058
   MLP, 5 medidas: acurácia balanceada 0.937 ± 0.014 | acurácia 0.944 | por semente: [0.9201 0.9376 0.9543]
   MLP, 3 medidas: acurácia balanceada 0.950 ± 0.003 | acurácia 0.952 | por semente: [0.9456 0.9536 0.9521]
   regra da chave: acurácia balanceada 0.718 | acurácia 0.401
só noites que não estão na base: n = 946
   MLP, 5 medidas: acurácia balanceada 0.957 ± 0.011 | acurácia 0.945 | por semente: [0.9431 0.9559 0.9711]
   MLP, 3 medidas: acurácia balanceada 0.968 ± 0.001 | acurácia 0.953 | por semente: [0.968  0.9695 0.9675]
   regra da chave: acurácia balanceada 0.731 | acurácia 0.382
já presentes na cópia de 01/10: n = 908
   MLP, 5 medidas: acurácia balanceada 0.947 ± 0.014 | acurácia 0.956 | por semente: [0.9301 0.9472 0.9635]
   MLP, 3 medidas: acurácia balanceada 0.960 ± 0.003 | acurácia 0.963 | por semente: [0.9548 0.9627 0.9611]
   regra da chave: acurácia balanceada 0.662 | acurácia 0.411
inéditas de 11/10: n = 150
   MLP, 5 medidas: acur

## Por espécie, e o erro principal

Sensibilidade da rede final em todas as gravações novas (média das 3 sementes), e quantas
gravações de *Myotis lavali* são chamadas de *Myotis nigricans*: nas inéditas de 11/10 e,
para comparar, no conjunto de teste.

In [15]:
colunas = variantes["5 medidas"]
sens_s = []
for semente in sementes:
    M = matriz_confusao(d_ext, prever(modelos[("5 medidas", semente)], X_ext[:, colunas]))
    sens_s.append(medidas(M)[2])
sens_media = np.mean(sens_s, axis=0)

print(f"{'espécie':24s} {'gravações':>10s} {'sensibilidade':>14s}")
for k in range(N_classes):
    print(f"{ESPECIES[k]:24s} {np.sum(d_ext == k):10d} {sens_media[k]:14.3f}")

# Myotis lavali (coluna 0) chamado de Myotis nigricans (linha 7)
k_lavali, k_nigricans = ESPECIES.index("Myotis_lavali"), ESPECIES.index("Myotis_nigricans")
for titulo, X_c, d_c in [("inéditas de 11/10", X_ext[e_inedita], d_ext[e_inedita]),
                         ("conjunto de teste", X_np[pos_te], d_np[pos_te])]:
    trocas = []
    for semente in sementes:
        M = matriz_confusao(d_c, prever(modelos[("5 medidas", semente)], X_c[:, colunas]))
        trocas.append(M[k_nigricans, k_lavali])
    n_lavali = np.sum(d_c == k_lavali)
    print(f"\n{titulo}: {np.mean(trocas):.1f} de {n_lavali} gravações de M. lavali chamadas de M. nigricans "
          f"({100 * np.mean(trocas) / n_lavali:.1f}%)")

espécie                   gravações  sensibilidade
Myotis_lavali                   795          0.943
Molossus_molossus                19          0.772
Lasiurus_blossevillii             4          1.000
Neoeptesicus_furinalis           62          0.925
Eumops_sp                         4          1.000
Molossus_currentium              34          0.971
Molossops_temminckii            128          0.971
Myotis_nigricans                 12          0.917

inéditas de 11/10: 16.7 de 121 gravações de M. lavali chamadas de M. nigricans (13.8%)

conjunto de teste: 33.7 de 1092 gravações de M. lavali chamadas de M. nigricans (3.1%)


## Resumo

- A rede final, sem nenhum ajuste, mantém o desempenho em gravações que entraram na planilha
  depois: a acurácia balanceada nas gravações de noites novas fica próxima da do teste.
- Essa medida varia mais entre sementes do que no teste, porque algumas espécies têm poucas
  gravações novas; e as gravações de noites novas vêm de poucas noites.
- O ponto fraco do teste reaparece, mais forte, nas gravações mais recentes: *Myotis lavali*
  chamado de *Myotis nigricans*.
- A variante de 3 medidas não perde para a rede de 5 medidas nas gravações novas.
- Nas checagens da base, ter uma vizinha no treino muda pouco a acurácia, e com noites
  inteiras fora do treino a acurácia balanceada fica próxima da obtida na divisão aleatória.